In [65]:
import random
import shutil 
from pathlib import Path
from cogent3 import load_aligned_seqs

# Set random seed for reproducibility
random.seed(41)

# Define your input and output directories
fasta_dir = Path("../data/hcm")  
output_dir = Path("./data")
json_outfile = output_dir / "human_chimp_pairs.json"

shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(exist_ok=True)
print (f"Clean output directory: {output_dir.resolve()}")

Clean output directory: /home/richard/source/BIOL8701-RichardMorris/final_presentation/experiments/1/data


# Sample 10 human-mouse-chimp orthologs

In [66]:
# Get all files matching HMP_*.fa
all_fasta_files = sorted(fasta_dir.glob("HMP_*.fa"))

# Sample 10 files
sampled_files = random.sample(all_fasta_files, 10)
sampled_files

[PosixPath('../data/hcm/HMP_ENSG00000143624.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000143156.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000127125.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000117448.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000143669.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000171357.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000189409.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000135775.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000163568.fa'),
 PosixPath('../data/hcm/HMP_ENSG00000134684.fa')]

# Extract degapped human-chimp sequence pairs to output folder

In [67]:
for fasta_path in sampled_files:
    # Load the aligned sequences
    sequences = load_aligned_seqs(fasta_path, moltype="dna")
    unaligned = sequences.degap()  

    # Extract just the human and chimp sequences
    # This assumes the labels include "homo sapiens" and "pan troglodytes"
    species_of_interest = ["homo sapiens", "pan troglodytes"]
    selected = [seq for name, seq in unaligned.named_seqs.items()
                if any(species in name.lower() for species in species_of_interest)]

    # If both species are present, write to file
    if len(selected) == 2:
        # Extract the Ensembl gene ID from the filename
        ensembl_id = fasta_path.stem.replace("HMP_", "")
        output_file = output_dir / f"{ensembl_id}_human_chimp.fa"
        # Write the selected sequences to file
        with open(output_file, "w") as f:
            for seq in selected:
                f.write(f">{seq.name}\n{str(seq)}\n")
    else:
        print(f"Skipping {fasta_path.name}: missing required species")
list(output_dir.glob("*_human_chimp.fa"))


[PosixPath('data/ENSG00000143624_human_chimp.fa'),
 PosixPath('data/ENSG00000143156_human_chimp.fa'),
 PosixPath('data/ENSG00000127125_human_chimp.fa'),
 PosixPath('data/ENSG00000117448_human_chimp.fa'),
 PosixPath('data/ENSG00000143669_human_chimp.fa'),
 PosixPath('data/ENSG00000171357_human_chimp.fa'),
 PosixPath('data/ENSG00000189409_human_chimp.fa'),
 PosixPath('data/ENSG00000135775_human_chimp.fa'),
 PosixPath('data/ENSG00000163568_human_chimp.fa'),
 PosixPath('data/ENSG00000134684_human_chimp.fa')]

# Create data file human_chimp_pairs.json

In [68]:
import json
from cogent3 import load_unaligned_seqs

# Initialize a dictionary to hold all the unaligned sequences
sequence_data = {}

# Load each pairwise alignment file
for fasta_path in sorted(output_dir.glob("*_human_chimp.fa")):
    ensembl_id = fasta_path.stem.replace("_human_chimp", "")
    sequences = load_unaligned_seqs(fasta_path, moltype="dna")

    # Extract sequences into a nested dictionary
    pair_dict = {}
    for name, seq in sequences.named_seqs.items():
        # Normalize to species name
        lowered = name.lower()
        if "homo sapiens" in lowered:
            pair_dict["homo sapiens"] = str(seq)
        elif "pan troglodytes" in lowered:
            pair_dict["pan troglodytes"] = str(seq)

    if set(pair_dict.keys()) == {"homo sapiens", "pan troglodytes"}:
        sequence_data[ensembl_id] = pair_dict
    else:
        print(f"Warning: Skipping {fasta_path.name}, incomplete pair")

# Save to a JSON file
with open(json_outfile, "w") as f:
    json.dump(sequence_data, f, indent=2)

print(f"Saved {len(sequence_data)} human-chimp pairs to {json_outfile}")


Saved 10 human-chimp pairs to data/human_chimp_pairs.json


# compute ungapped Smith-Waterman alignment and store to human_chimp_pairs.json

In [69]:
import json
from madb import smith_waterman_ungapped
from pathlib import Path

with open(json_outfile, "r") as f:
    sequence_data = json.load(f)

# Add alignment result for each Ensembl ID
for ensembl_id, pair in sequence_data.items():
    human = pair.get("homo sapiens")
    chimp = pair.get("pan troglodytes")

    if human and chimp:
        score, aligned_human, aligned_chimp = smith_waterman_ungapped(human, chimp)
        pair["ungapped_alignment"] = {
            "score": score,
            "homo sapiens": aligned_human,
            "pan troglodytes": aligned_chimp,
            "length" : len(aligned_human)
        }
    else:
        print(f"Missing sequences for {ensembl_id}")

# Save the enriched JSON back to the same path (overwrite)
with open(json_outfile, "w") as f:
    json.dump(sequence_data, f, indent=2)

print(f"Updated alignments with Smith-Waterman results saved to {json_outfile}")


Updated alignments with Smith-Waterman results saved to data/human_chimp_pairs.json


# compute longest madb braid and store to human_chimp_pairs.json

In [70]:
import json
from pathlib import Path
from madb import make_graph  

# Load the JSON
with open(json_outfile) as f:
    sequence_data = json.load(f)

# Process each entry to add the longest braid
for ensembl_id, entry in sequence_data.items():
    try:
        # Extract just the sequences
        seqs = {
            k: v for k, v in entry.items()
            if k in {"homo sapiens", "pan troglodytes"} and isinstance(v, str)
        }

        if not all(sp in seqs for sp in ["homo sapiens", "pan troglodytes"]):
            entry["braid"] = None
            continue

        dbg = make_graph(seqs, kmer_size=12)  # Only pass the filtered name→sequence mapping
        

        longest_braid = dbg.longest_braid()

        if longest_braid and longest_braid.length > 0:
            entry["braid"] = {
                "start_kmer": longest_braid.start.kmer,
                "end_kmer": longest_braid.end.kmer,
                "length": longest_braid.length,
                "sequences": longest_braid.sequence_fragments
            }
        else:
            entry["braid"] = None

    except Exception as e:
        print(f"Error processing {ensembl_id}: {e}")
        entry["braid"] = None

with open(json_outfile, "w") as f:
    json.dump(sequence_data, f, indent=2)

print(f"Braid info added to {json_outfile}")

from pprint import pprint
# Show one with a braid
pprint(next((v for v in sequence_data.values() if v["braid"]), None))


Error processing ENSG00000163568: Sequence index '2' not found in end node
Braid info added to data/human_chimp_pairs.json
{'braid': {'end_kmer': '',
           'length': 918,
           'sequences': {'homo sapiens': 'TTGGTCTGGGTACCTGGAAGAGTGAGCCTGGTCAGGTAAAAGCAGCTGTTAAGTATGCCCTTAGCGTAGGCTACCGCCACATTGATTGTGCTGCTATCTACGGCAATGAGCCTGAGATTGGGGAGGCCCTGAAGGAGGACGTGGGACCAGGCAAGGCGGTGCCTCGGGAGGAGCTGTTTGTGACATCCAAGCTGTGGAACACCAAGCACCACCCCGAGGATGTGGAGCCTGCCCTCCGGAAGACTCTGGCTGACCTCCAGCTGGAGTATCTGGACCTGTACCTGATGCACTGGCCTTATGCCTTTGAGCGGGGAGACAACCCCTTCCCCAAGAATGCTGATGGGACTATATGCTACGACTCCACCCACTACAAGGAGACTTGGAAGGCTCTGGAGGCACTGGTGGCTAAGGGGCTGGTGCAGGCGCTGGGCCTGTCCAACTTCAACAGTCGGCAGATTGATGACATACTCAGTGTGGCCTCCGTGCGTCCAGCTGTCTTGCAGGTGGAATGCCACCCATACTTGGCTCAAAATGAGCTAATTGCCCACTGCCAAGCACGTGGCCTGGAGGTAACTGCTTATAGCCCTTTGGGCTCCTCTGATCGTGCATGGCGTGATCCTGATGAGCCTGTCCTGCTGGAGGAACCAGTAGTCCTGGCATTGGCTGAAAAGTATGGCCGATCTCCAGCTCAGATCTTGCTCAGGTGGCAGGTCCAGCGGAAAGTGATCTGCATCCCCAAAAGTATCACTCCTTCTCGAATCCTTCAGAACATCAAGGTGTTT

In [100]:
import json
import numpy as np
import plotly.graph_objects as go

# Load JSON data
with open(json_outfile) as f:
    alignment_data = json.load(f)

# Collect values
braid_lengths = []
sw_lengths = []
ensembl_ids = []

for ensembl_id, entry in alignment_data.items():
    braid = entry.get("braid")
    sw = entry.get("ungapped_alignment")
    if braid and sw and "length" in braid and "length" in sw:
        braid_lengths.append(braid["length"])
        sw_lengths.append(sw["length"])
        ensembl_ids.append(ensembl_id)

# Convert to numpy arrays
x = np.array(sw_lengths)
y = np.array(braid_lengths)
ensembl_ids = np.array(ensembl_ids)

# Pearson correlation
correlation = np.corrcoef(x, y)[0, 1]

# Linear regression
slope, intercept = np.polyfit(x, y, 1)
regression_y = slope * x + intercept

# Axis range
max_val = max(x.max(), y.max()) * 1.05

# Build figure
fig = go.Figure()

# Scatter plot of all points
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="markers",
    showlegend=False,
    marker=dict(size=14, color="dodgerblue", opacity=0.7),
    text=ensembl_ids,
    hovertemplate="Ensembl ID: %{text}<br>SW Length: %{x}<br>Braid Length: %{y}<extra></extra>"
))

# Dotted expected line (y = x)
fig.add_trace(go.Scatter(
    x=[0, max_val],
    y=[0, max_val],
    mode="lines",
    name="Expected",
    line=dict(dash="dot", color="gray", width=4)
))

# Regression line
fig.add_trace(go.Scatter(
    x=x,
    y=regression_y,
    mode="lines",
    name="Regression fit",
    line=dict(color="crimson", width=4)
))

# Floating Pearson r label
fig.update_layout(
    xaxis_title="Ungapped Smith-Waterman",
    yaxis_title="Longest Braid (k=12)",
    width=800,
    height=800,
    font=dict(size=24),
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=18)
    ),
    annotations=[
        dict(
            x=0,
            y=0.90 * max_val,
            xref="x",
            yref="y",
            text=f"Pearson r = {correlation:.3f}",
            showarrow=False,
            font=dict(size=18, color="black"),
            align="left",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            borderpad=4,
            opacity=0.95,
            xshift=100
        ),
    ]
)

# Format axes with 'k' notation and ensure equal scale
fig.update_xaxes(
    range=[0, 10_000],
    scaleanchor="y",
    scaleratio=1,
    tickformat=".0~s",  # e.g. 2000 → 2k
)
fig.update_yaxes(
    range=[0, 10_000],
    tickformat=".0~s"
)

# Export image
fig.write_image("../../slideshows/images/braid_vs_sw.png", width=800, height=800)
fig.show()


In [72]:
# Cells that have been tagged for export can be exported with nbtools.export_cells

import nbtools
nbtools.export_cells('./experiment1.ipynb', '../../slideshows/images')


💻 Processing: ./experiment1.ipynb


In [73]:

from cogent3.core.alignment import Alignment
from cogent3 import get_app

aligner = get_app("progressive_align",model="JC69")

unaln = load_unaligned_seqs("./data/ENSG00000116251_human_chimp.fa", moltype="dna")
aligned = aligner(unaln)
aligned.set_repr_policy(num_pos=10_000)
aligned

FileNotFoundError: [Errno 2] No such file or directory: 'data/ENSG00000116251_human_chimp.fa'

In [ ]:
from madb.debruijngraph import load_graph


dbg = make_graph(unaln, kmer_size=12)
dbg.longest_braid()


{'homo sapiens': 'TTAATCCTCGTCTTCCTCCTCTTCTTCGTCCTGGTTAATCTGGAAGTAACGTAATTCGTAACTCTCTTTGCTGTTAGCAACTACGCGCAACCAGTCACGTAGATTATTCTTCTTCAAATATTTTTTGGTGAGATATTTCAAATACCTTTTGGAGAAAGGCACCTCGGATGTCACGGTGATCTTGCTCTTGCTCCTTTCGATGGTCACCACCCCTCCACCAAGGTTCCCAGCTTTTCCGTTCACTTTGATCCTTTCTTGCAAAAACTGCTCAAAATTGGCAGCATCCATGATTCCATCTTCTACAGGGTGGGTGCAATCAAGAGTGAACTTCAGAACTTGCTTCTTTTTTTTGCCCCCCTTCACCACAAGCTTTTTC'}

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Generate length values
length = np.linspace(0, 10, 200)

# Define time as a function of length
time_linear = 2 * length
time_quadratic = length ** 2

# Create the figure
fig = go.Figure()

# Add y = 2x (dotted)
fig.add_trace(go.Scatter(
    x=length,
    y=time_linear,
    mode="lines",
    name="Time = 2 × Length",
    line=dict(color="blue", width=4, dash="dot")
))

# Add y = x^2 (solid)
fig.add_trace(go.Scatter(
    x=length,
    y=time_quadratic,
    mode="lines",
    name="Time = Length²",
    line=dict(color="red", width=4)
))

# Layout
fig.update_layout(
    title="Linear vs Quadratic Time Growth",
    xaxis_title="Length",
    yaxis_title="Time",
    font=dict(size=24),
    width=800,
    height=600,
    legend=dict(x=0.02, y=0.98, font=dict(size=24)),
    margin=dict(l=60, r=40, t=60, b=60)
)
fig.write_image("../../slideshows/images/linear_vs_quadratic.png", width=800, height=800)
fig.show()
